# rustwood vs LightGBM — Colab demo

[`rustwood`](https://github.com/advpropsys/rustwood) is a GPU oblivious-tree gradient
booster whose CUDA kernels are pure Rust (compiled to PTX via cuda-oxide). It ships a
thin Python API (GPU **and** CPU), a GPU-free `--device cpu` trainer, and an instant
`.rwood` model format.

This notebook uses the Python API to train rustwood and LightGBM on an sklearn dataset
and compares **training time, accuracy, and model size**.

**Requirements:** a GPU runtime (`Runtime -> Change runtime type -> GPU`). The one-time
build takes ~10-15 min (it compiles the cuda-oxide backend).


## 1. Build rustwood + install the Python package

`advpropsys/rustwood` is private — paste a GitHub token with read access (or make the
repo public and leave it blank).


In [ ]:
GITHUB_TOKEN = ""  #@param {type:"string"}
REPO = "advpropsys/rustwood"


In [ ]:
import os, subprocess, time

# Rust toolchain (the repo pins the exact nightly via rust-toolchain.toml).
!curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --default-toolchain none >/dev/null 2>&1
os.environ['PATH'] = '/root/.cargo/bin:' + os.environ['PATH']

# Clone (recursive: pulls the cuda-oxide submodule).
url = f'https://{GITHUB_TOKEN}@github.com/{REPO}.git' if GITHUB_TOKEN else f'https://github.com/{REPO}.git'
subprocess.run(['rm', '-rf', '/content/rustwood'])
assert subprocess.run(['git','clone','--recursive','-q',url,'/content/rustwood']).returncode == 0, 'clone failed (token?)'
os.chdir('/content/rustwood')

# Build for this GPU's compute capability (Colab is usually a T4 = sm_75).
cap = subprocess.check_output(['nvidia-smi','--query-gpu=compute_cap','--format=csv,noheader']).decode().split('\n')[0].strip()
ARCH = 'sm_' + cap.replace('.', '')
print('GPU compute capability', cap, '->', ARCH)
t = time.time()
!cd external/cuda-oxide && cargo build -q -p cargo-oxide
!ARCH={ARCH} CUDA_PATH=/usr/local/cuda ./build.sh 2>&1 | tail -2
assert os.path.exists('target/release/rustwood'), 'build failed'
print(f'built in {time.time()-t:.0f}s')


In [ ]:
# Install the thin Python wrapper and point it at the binary we just built.
!pip install -q ./python
os.environ['RUSTWOOD_BIN'] = os.path.abspath('target/release/rustwood')
os.chdir('/content')
import rustwood; print('rustwood Python API ready ->', rustwood.find_binary())


## 2. Data (sklearn California Housing)


In [ ]:
import numpy as np
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
X, y = fetch_california_housing(return_X_y=True)
Xtr, Xte, ytr, yte = train_test_split(X.astype('f4'), y.astype('f4'), test_size=0.2, random_state=0)
print('train', Xtr.shape, 'test', Xte.shape)


## 3. rustwood — clean Python API


In [ ]:
import time, os
from rustwood import RustwoodRegressor, load
from sklearn.metrics import r2_score

t = time.perf_counter()
m = RustwoodRegressor(n_trees=500, depth=6, learning_rate=0.1, device='gpu').fit(Xtr, ytr)
rw_train = time.perf_counter() - t
rw_r2 = r2_score(yte, m.predict(Xte))

# save / load round-trip (the .rwood format)
m.save('/content/model.rwood')
m2 = load('/content/model.rwood')
assert abs(r2_score(yte, m2.predict(Xte)) - rw_r2) < 1e-6  # identical predictions
rw_size = os.path.getsize('/content/model.rwood') / 1024
print(f'rustwood  train={rw_train:.2f}s  R2={rw_r2:.4f}  model={rw_size:.0f} KB  (save/load round-trips)')


## 4. LightGBM


In [ ]:
import lightgbm as lgb, time, os
m_lgb = lgb.LGBMRegressor(n_estimators=500, max_depth=6, num_leaves=64,
                          learning_rate=0.1, verbose=-1)
t = time.perf_counter(); m_lgb.fit(Xtr, ytr); lgb_train = time.perf_counter() - t
lgb_r2 = r2_score(yte, m_lgb.predict(Xte))
m_lgb.booster_.save_model('/content/model.txt')
lgb_size = os.path.getsize('/content/model.txt') / 1024
print(f'LightGBM  train={lgb_train:.2f}s  R2={lgb_r2:.4f}  model={lgb_size:.0f} KB')


## 5. Comparison


In [ ]:
import matplotlib.pyplot as plt
print(f'{"":10}{"train_s":>9}{"R2":>8}{"model_KB":>10}')
print(f'{"rustwood":10}{rw_train:>9.2f}{rw_r2:>8.4f}{rw_size:>10.0f}')
print(f'{"LightGBM":10}{lgb_train:>9.2f}{lgb_r2:>8.4f}{lgb_size:>10.0f}')

fig, ax = plt.subplots(1, 3, figsize=(11, 3.2))
L = ['rustwood', 'LightGBM']; C = ['#E8613C', '#5FA08C']
ax[0].bar(L, [rw_train, lgb_train], color=C); ax[0].set_title('train time (s)')
ax[1].bar(L, [rw_r2, lgb_r2], color=C); ax[1].set_title('test R2'); ax[1].set_ylim(0.7, 0.9)
ax[2].bar(L, [rw_size, lgb_size], color=C); ax[2].set_title('model size (KB)')
for a in ax: a.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.show()


## Notes

- rustwood uses **oblivious (symmetric) trees** — on this all-numeric dataset LightGBM's
  leaf-wise trees may edge it on accuracy (the structural trade-off), while rustwood wins
  on **training speed** and ships a far **smaller, instant-loading `.rwood` model**.
- The Python API is a thin pass-through to the binary (GPU + CPU); it adds no overhead
  beyond writing the input arrays. Use `device='cpu'` for a GPU-free run.
- The whole library is ~3.2k lines of Rust (vs XGBoost 87k / LightGBM 63k C++/CUDA).
